# Regression Backtest

In [62]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from   matplotlib.ticker import FuncFormatter

# Get Data

In [2]:
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_path = os.path.join(root_path, "data")
raw_path  = os.path.join(data_path, "RawData")
tsy_path  = os.path.join(raw_path, "TreasuryFutures.parquet")

process_path = os.path.join(data_path, "ProcessedData")
date_path    = os.path.join(process_path, "OrderedDates.parquet")

In [3]:
df_tsy   = pd.read_parquet(path = tsy_path, engine = "pyarrow")
df_dates = pd.read_parquet(path = date_path, engine = "pyarrow")

In [4]:
processed_path = os.path.join(process_path, "ProcessedSentimentData.parquet")
df_zscore      = (pd.read_parquet(
    path = processed_path, engine = "pyarrow")
    [["date", "security", "lag_zscore"]])

In [42]:
df_zscore_tmp = (df_dates.query(
    "day == day.min()")
    [["date", "event"]].
    drop_duplicates().
    merge(right = df_zscore, how = "inner", on = ["date"]))

In [70]:
df_fomc_rtn = (df_tsy.assign(
    date = lambda x: pd.to_datetime(x.date).dt.date).
    merge(right = df_dates, how = "inner", on = ["date"]).
    assign(security = lambda x: x.security.str.split("1").str[0].str.strip()).
    pivot(index = "day", columns = ["event", "security"], values = "PX_bps").
    cumsum().
    ffill().
    reset_index().
    melt(id_vars = [("day", "")], value_name = "rtn").
    rename(columns = {("day", ""): "day"}).
    query("day == day.max()").
    drop(columns = ["day"]))

In [71]:
df_panel = (df_zscore_tmp.pivot(
    index = "event", columns = "security", values = "lag_zscore").
    reset_index().
    merge(right = df_fomc_rtn, how = "inner", on = ["event"]))

# OLS Regression